hw3_RLLib.ipynb


this doesnt work. the code for cartpole and the traffic simulator is under hw3_rllib. Difficult to install rllib, requires --force flag for python3.12 else it wont install.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Single Colab cell: RLlib + EnvPool CartPole with save/restore + eval video
# Fix:
# - pass a gym.Env subclass directly to RLlib
# - wrap EnvPool(num_envs=1) so RLlib sees a single non-vectorized env

!pip -q install "ray[rllib]" envpool gymnasium moviepy imageio imageio-ffmpeg

import os
import glob
import shutil
import numpy as np

import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

import ray
from ray.rllib.algorithms.ppo import PPOConfig

import envpool


# ============================================================
# Paths / config
# ============================================================

BASE_DIR = "/content/rllib_envpool_cartpole"
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
VIDEO_DIR = os.path.join(BASE_DIR, "videos")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

TRAIN_ITERS = 8
SEED = 0
ENV_NAME = "CartPole-v1"


# ============================================================
# Non-vectorized adapter around EnvPool(num_envs=1)
# ============================================================

class EnvPoolCartPoleSingle(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, config=None):
        config = config or {}
        seed = config.get("seed", SEED)

        self.seed_value = seed
        self.env = envpool.make(
            ENV_NAME,
            env_type="gym",
            num_envs=1,
            seed=seed,
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(4,),
            dtype=np.float32,
        )
        self.action_space = spaces.Discrete(2)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.seed_value = seed
            self.env = envpool.make(
                ENV_NAME,
                env_type="gym",
                num_envs=1,
                seed=seed,
            )

        out = self.env.reset()
        if isinstance(out, tuple) and len(out) == 2:
            obs, info = out
        else:
            obs, info = out, {}

        obs = np.asarray(obs)[0].astype(np.float32)
        return obs, info

    def step(self, action):
        action_batch = np.asarray([action], dtype=np.int64)
        obs, reward, done, info = self.env.step(action_batch)

        obs = np.asarray(obs)[0].astype(np.float32)
        reward = float(np.asarray(reward)[0])
        done = bool(np.asarray(done)[0])

        terminated = done
        truncated = False

        if isinstance(info, dict):
            info0 = {}
            for k, v in info.items():
                try:
                    info0[k] = np.asarray(v)[0]
                except Exception:
                    info0[k] = v
        else:
            info0 = {}

        return obs, reward, terminated, truncated, info0


# ============================================================
# Ray init
# ============================================================

ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)


# ============================================================
# RLlib config
# ============================================================

config = (
    PPOConfig()
    .environment(
        EnvPoolCartPoleSingle,
        env_config={"seed": SEED},
    )
    .framework("torch")
    .env_runners(
        num_env_runners=0,
        num_envs_per_env_runner=1,
    )
)

algo = config.build_algo()

print("Training...")
for i in range(TRAIN_ITERS):
    result = algo.train()

    ep_ret = None
    for key_path in [
        ("env_runners", "episode_return_mean"),
        ("evaluation", "env_runners", "episode_return_mean"),
    ]:
        cur = result
        ok = True
        for k in key_path:
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if ok:
            ep_ret = cur
            break

    print(f"iter={i+1} episode_return_mean={ep_ret}")


# ============================================================
# Save checkpoint
# ============================================================

checkpoint_path = algo.save_to_path(CKPT_DIR)
print("Saved checkpoint:", checkpoint_path)


# ============================================================
# Restore into a fresh Algorithm
# ============================================================

algo_restored = config.build_algo()
algo_restored.restore_from_path(checkpoint_path)
print("Restored checkpoint into new PPO Algorithm.")


# ============================================================
# Record evaluation video using plain Gymnasium CartPole
# ============================================================

def latest_mp4(video_dir: str):
    files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
    return files[-1] if files else None


eval_video_dir = os.path.join(VIDEO_DIR, "eval")
if os.path.exists(eval_video_dir):
    shutil.rmtree(eval_video_dir)
os.makedirs(eval_video_dir, exist_ok=True)

env = gym.make(ENV_NAME, render_mode="rgb_array")
env = RecordVideo(
    env,
    video_folder=eval_video_dir,
    episode_trigger=lambda episode_id: True,
    name_prefix="cartpole_eval",
)

obs, info = env.reset(seed=SEED)
done = False
ep_return = 0.0

while not done:
    action = algo_restored.compute_single_action(obs, explore=False)
    if isinstance(action, tuple):
        action = action[0]
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    ep_return += reward

env.close()

video_path = latest_mp4(eval_video_dir)
print("Eval return:", ep_return)
print("Video path:", video_path)

if video_path is not None:
    display(Video(video_path, embed=True))

/usr/local/lib/python3.12/dist-packages/tensorflow_probability/python/__init__.py:57: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if (distutils.version.LooseVersion(tf.__version__) <
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is dep

Training...
iter=1 episode_return_mean=20.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


iter=2 episode_return_mean=36.07


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

iter=8 episode_return_mean=250.1764705882353
Saved checkpoint: /content/rllib_envpool_cartpole/checkpoints


/usr/local/lib/python3.12/dist-packages/ray/rllib/algorithms/algorithm.py:527: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/usr/local/lib/python3.12/dist-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/usr/local/lib/python3.12/dist-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppre

Restored checkpoint into new PPO Algorithm.


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

AttributeError: 'SingleAgentEnvRunner' object has no attribute 'get_policy'